# Gene-Environment Association Analysis

For species with strains isolated from diverse environments, test whether individual accessory gene clusters are significantly over- or under-represented in specific environments.

**Statistical approach**:
- For each accessory/singleton gene cluster, build a contingency table: present/absent x environment categories
- Fisher's exact test (2 categories) or chi-squared test (>2 categories)
- Benjamini-Hochberg FDR correction across all clusters per species
- Report gene clusters with q-value < 0.05

**Depends on**: `data/genome_env_metadata.csv` from notebook 02.

**Portability**: Gene cluster presence queries touch the billion-row `gene_genecluster_junction` table and run best on Spark. The statistical analysis itself runs locally in pandas/scipy.

## 1. Setup

In [1]:
from berdl_notebook_utils import get_spark_session
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact, chi2_contingency
from statsmodels.stats.multitest import multipletests

spark = get_spark_session("GeneEnvAssociation")

def refresh_spark():
    """Restart Spark session to reset auth token.

    Workaround for KBaseAuthServerInterceptor timeout (~40 min)
    that terminates long-running Spark Connect gRPC sessions.
    See nmdc_flattened_biosample_loading.md for details.
    """
    global spark
    try:
        spark.stop()
    except Exception:
        pass
    spark = get_spark_session("GeneEnvAssociation")

# Load environment metadata from notebook 02
genome_env = pd.read_csv('../data/genome_env_metadata.csv')
print(f"Loaded {len(genome_env)} genomes with environment metadata")
print(f"Environment categories:\n{genome_env['env_category'].value_counts().to_string()}")

Loaded 292913 genomes with environment metadata
Environment categories:
env_category
human_clinical      104932
other                81829
host_associated      51868
aquatic              33834
soil                  7620
food                  3998
sediment              3353
engineered            3324
plant_associated      1507
air                    648


## 2. Species selection

Select species with enough genomes in multiple environment categories for statistical testing. Criteria: >= 2 environment categories with >= 10 genomes each.

In [2]:
env_filtered = genome_env[genome_env['env_category'] != 'other'].copy()

species_env = env_filtered.groupby(
    ['gtdb_species_clade_id', 'env_category']
).size().reset_index(name='n_genomes')

# Species with >= 2 categories having >= 10 genomes each
species_qualifying = species_env[species_env['n_genomes'] >= 10].groupby(
    'gtdb_species_clade_id'
).agg(
    n_qualifying_categories=('env_category', 'count'),
    total_qualifying_genomes=('n_genomes', 'sum'),
    categories=('env_category', lambda x: ', '.join(sorted(x)))
).reset_index()

species_qualifying = species_qualifying[
    species_qualifying['n_qualifying_categories'] >= 2
].sort_values('total_qualifying_genomes', ascending=False)

print(f"Species meeting criteria (>=2 categories with >=10 genomes): {len(species_qualifying)}")
species_qualifying.head(20)

Species meeting criteria (>=2 categories with >=10 genomes): 212


,gtdb_species_clade_id,n_qualifying_categories,total_qualifying_genomes,categories
1261,s__Staphylococcus_aureus--RS_GCF_001027105.1,5,12906,"air, aquatic, food, host_associated, human_cli..."
680,s__Klebsiella_pneumoniae--RS_GCF_000742135.1,6,12888,"aquatic, engineered, food, host_associated, hu..."
1322,s__Streptococcus_pneumoniae--RS_GCF_001457635.1,2,7933,"host_associated, human_clinical"
1229,s__Salmonella_enterica--RS_GCF_000006945.2,8,6838,"air, aquatic, food, host_associated, human_cli..."
878,s__Mycobacterium_tuberculosis--RS_GCF_000195955.2,2,6630,"host_associated, human_clinical"
19,s__Acinetobacter_baumannii--RS_GCF_009759685.1,3,5748,"aquatic, host_associated, human_clinical"
1113,s__Pseudomonas_aeruginosa--RS_GCF_001457615.1,4,4727,"aquatic, host_associated, human_clinical, soil"
487,s__Enterobacter_hormaechei_A--RS_GCF_001729745.1,3,2171,"aquatic, host_associated, human_clinical"
1324,s__Streptococcus_pyogenes--RS_GCF_002055535.1,2,2031,"host_associated, human_clinical"
312,s__Campylobacter_D_jejuni--RS_GCF_001457695.1,4,1957,"aquatic, food, host_associated, human_clinical"


In [3]:
# Get pangenome statistics for qualifying species
if len(species_qualifying) > 0:
    species_list = "','".join(species_qualifying['gtdb_species_clade_id'].tolist()[:50])

    pangenome_stats = spark.sql(f"""
        SELECT
            s.GTDB_species,
            s.gtdb_species_clade_id,
            p.no_genomes,
            p.no_core,
            p.no_gene_clusters,
            p.no_aux_genome,
            p.no_singleton_gene_clusters
        FROM kbase_ke_pangenome.pangenome p
        JOIN kbase_ke_pangenome.gtdb_species_clade s
          ON p.gtdb_species_clade_id = s.gtdb_species_clade_id
        WHERE s.gtdb_species_clade_id IN ('{species_list}')
        ORDER BY p.no_genomes DESC
    """).toPandas()

    target_species = pangenome_stats.merge(species_qualifying, on='gtdb_species_clade_id')
    print(f"Target species with pangenome stats: {len(target_species)}")
    target_species[['GTDB_species', 'no_genomes', 'no_gene_clusters', 'no_aux_genome',
                    'n_qualifying_categories', 'categories']].head(15)

Target species with pangenome stats: 50


## 3. Gene cluster presence matrix

For each target species, build a presence/absence matrix of accessory gene clusters. Core clusters are excluded (they're present in all genomes and uninformative for this test).

In [4]:
def get_accessory_presence(species_id, genome_ids):
    """Get accessory gene cluster presence for genomes of a species.

    Returns DataFrame with columns: genome_id, gene_cluster_id
    """
    genome_str = "','".join(genome_ids)

    df = spark.sql(f"""
        SELECT DISTINCT
            g.genome_id,
            gg.gene_cluster_id
        FROM kbase_ke_pangenome.gene g
        JOIN kbase_ke_pangenome.gene_genecluster_junction gg
            ON g.gene_id = gg.gene_id
        JOIN kbase_ke_pangenome.gene_cluster gc
            ON gg.gene_cluster_id = gc.gene_cluster_id
        WHERE gc.gtdb_species_clade_id = '{species_id}'
          AND gc.is_core = false
          AND g.genome_id IN ('{genome_str}')
    """).toPandas()

    return df

# Test with the first target species
if len(target_species) > 0:
    test_species = target_species.iloc[0]
    test_species_id = test_species['gtdb_species_clade_id']
    test_genomes = env_filtered[
        env_filtered['gtdb_species_clade_id'] == test_species_id
    ]['genome_id'].tolist()

    print(f"Testing with: {test_species['GTDB_species']}")
    print(f"  Genomes with env labels: {len(test_genomes)}")

    presence = get_accessory_presence(test_species_id, test_genomes)
    print(f"  Accessory gene-genome pairs: {len(presence):,}")
    print(f"  Unique gene clusters: {presence['gene_cluster_id'].nunique():,}")

Testing with: s__Staphylococcus_aureus
  Genomes with env labels: 12913
  Accessory gene-genome pairs: 7,304,107
  Unique gene clusters: 133,262


## 4. Gene-environment association testing

For each accessory gene cluster, test whether its prevalence differs across environment categories.

In [5]:
def test_gene_env_associations(presence_df, env_df, species_id, min_per_cat=10):
    """Test gene-environment associations for one species.

    For each accessory gene cluster, build a contingency table of
    present/absent x environment category and run Fisher's or chi-squared test.
    """
    species_genomes = env_df[
        (env_df['gtdb_species_clade_id'] == species_id) &
        (env_df['env_category'] != 'other')
    ][['genome_id', 'env_category']].copy()

    cat_counts = species_genomes['env_category'].value_counts()
    valid_cats = cat_counts[cat_counts >= min_per_cat].index.tolist()

    if len(valid_cats) < 2:
        return pd.DataFrame()

    species_genomes = species_genomes[species_genomes['env_category'].isin(valid_cats)]
    all_genome_set = set(species_genomes['genome_id'])

    gene_genomes = presence_df.groupby('gene_cluster_id')['genome_id'].apply(set).to_dict()

    results = []
    for gc_id, genomes_with_gene in gene_genomes.items():
        genomes_with_gene = genomes_with_gene & all_genome_set
        if len(genomes_with_gene) == 0:
            continue

        table = []
        cat_prevalences = {}
        for cat in valid_cats:
            cat_genomes = set(species_genomes[
                species_genomes['env_category'] == cat
            ]['genome_id'])
            present = len(genomes_with_gene & cat_genomes)
            absent = len(cat_genomes - genomes_with_gene)
            table.append([present, absent])
            total = present + absent
            cat_prevalences[cat] = present / total if total > 0 else 0

        table = np.array(table)
        if table[:, 0].sum() == 0 or table[:, 1].sum() == 0:
            continue

        if len(valid_cats) == 2:
            odds_ratio, p_val = fisher_exact(table)
            stat = odds_ratio
            test_type = 'fisher'
        else:
            try:
                chi2, p_val, dof, expected = chi2_contingency(table)
                stat = chi2
                test_type = 'chi2'
            except ValueError:
                continue

        max_cat = max(cat_prevalences, key=cat_prevalences.get)

        results.append({
            'gene_cluster_id': gc_id,
            'test_type': test_type,
            'test_statistic': stat,
            'p_value': p_val,
            'prevalence_diff': max(cat_prevalences.values()) - min(cat_prevalences.values()),
            'max_prevalence': max(cat_prevalences.values()),
            'min_prevalence': min(cat_prevalences.values()),
            'enriched_in': max_cat,
            'n_genomes_with_gene': len(genomes_with_gene),
        })

    results_df = pd.DataFrame(results)

    if len(results_df) > 0:
        reject, q_values, _, _ = multipletests(
            results_df['p_value'], method='fdr_bh', alpha=0.05
        )
        results_df['q_value'] = q_values
        results_df['significant'] = reject

    return results_df

In [6]:
%%time

# Run for the test species
if len(target_species) > 0:
    results = test_gene_env_associations(presence, genome_env, test_species_id)

    if len(results) > 0:
        n_sig = results['significant'].sum()
        n_total = len(results)
        print(f"Results for {test_species['GTDB_species']}:")
        print(f"  Gene clusters tested: {n_total:,}")
        print(f"  Significant (q < 0.05): {n_sig} ({100*n_sig/n_total:.1f}%)")
        print(f"\nTop hits by q-value:")
        print(results.sort_values('q_value').head(10)[
            ['gene_cluster_id', 'prevalence_diff', 'enriched_in', 'q_value']
        ].to_string(index=False))

Results for s__Staphylococcus_aureus:
  Gene clusters tested: 133,007
  Significant (q < 0.05): 13351 (10.0%)

Top hits by q-value:
          gene_cluster_id  prevalence_diff     enriched_in  q_value
NZ_JACYHK010000001.1_1571         0.575337  human_clinical      0.0
      NZ_LLBV01000009.1_3         0.189241 host_associated      0.0
   NZ_JAEADA010000033.1_1         0.466667             air      0.0
      NZ_RQSV01000329.1_1         0.603650  human_clinical      0.0
  NZ_JAFEPH010000022.1_12         0.166667         aquatic      0.0
  NZ_JAFEPH010000022.1_10         0.166667         aquatic      0.0
  NZ_JAFEPH010000022.1_11         0.166667         aquatic      0.0
 NZ_JAEADP010000026.1_101         0.516667             air      0.0
     NZ_FMRQ01000009.1_64         0.215594 host_associated      0.0
      NZ_CFPL01000044.1_2         0.576314  human_clinical      0.0
CPU times: user 20min 38s, sys: 1.95 s, total: 20min 40s
Wall time: 20min 37s


## 5. Collect results

S. aureus (cells 3-4) serves as proof of concept that the framework works. The full multi-species loop is deferred — queries for species with 10K+ genomes can exceed the ~40-minute Spark Connect auth timeout even with session refresh.

In [ ]:
# Use S. aureus results from cells 3-4 as the example species.
# The 10-species loop is deferred: single queries for large species
# (12K+ genomes joining gene_genecluster_junction) can exceed the
# ~40-min Spark Connect auth timeout even with session refresh.
all_results = {}

if len(results) > 0:
    species_name = test_species['GTDB_species']
    all_results[species_name] = results
    n_sig = results['significant'].sum()
    n_total = len(results)
    print(f"Example species: {species_name}")
    print(f"  Gene clusters tested: {n_total:,}")
    print(f"  Significant (q < 0.05): {n_sig} ({100*n_sig/n_total:.1f}%)")
    print(f"\n  Environment categories tested:")
    print(f"  {test_species['categories']}")
    print(f"\n  Enrichment by environment:")
    print(results[results['significant']]['enriched_in'].value_counts().to_string())

## 6. Functional enrichment of environment-associated genes

For significant gene clusters (q < 0.05), retrieve eggNOG annotations and test for COG category enrichment vs all accessory genes.

In [ ]:
def get_cluster_annotations(cluster_ids):
    """Get COG and KEGG annotations for a set of gene clusters."""
    if len(cluster_ids) == 0:
        return pd.DataFrame()

    cluster_str = "','".join(cluster_ids)
    return spark.sql(f"""
        SELECT
            e.query_name as gene_cluster_id,
            e.Preferred_name,
            e.COG_category,
            e.Description,
            e.KEGG_ko,
            e.KEGG_Pathway,
            e.EC
        FROM kbase_ke_pangenome.eggnog_mapper_annotations e
        WHERE e.query_name IN ('{cluster_str}')
          AND e.COG_category IS NOT NULL
          AND e.COG_category != '-'
    """).toPandas()

# Refresh after the long b5 loop
refresh_spark()

enrichment_results = {}

for species_name, results in all_results.items():
    sig_clusters = results[results['significant']]['gene_cluster_id'].tolist()
    all_clusters = results['gene_cluster_id'].tolist()

    if len(sig_clusters) == 0:
        continue

    sig_annot = get_cluster_annotations(sig_clusters)
    bg_annot = get_cluster_annotations(all_clusters[:5000])  # sample background

    if len(sig_annot) == 0 or len(bg_annot) == 0:
        continue

    # Explode multi-letter COG categories
    sig_cogs = list(''.join(sig_annot['COG_category'].tolist()))
    bg_cogs = list(''.join(bg_annot['COG_category'].tolist()))

    sig_cog_counts = pd.Series(sig_cogs).value_counts()
    bg_cog_counts = pd.Series(bg_cogs).value_counts()

    enrichment_results[species_name] = {
        'sig_cogs': sig_cog_counts,
        'bg_cogs': bg_cog_counts,
        'n_sig': len(sig_clusters),
        'n_total': len(all_clusters)
    }

    print(f"\n{species_name}:")
    print(f"  Significant clusters with COG: {len(sig_annot)}")
    print(f"  Top COGs in significant set: {sig_cog_counts.head(5).to_dict()}")

## 7. Volcano plot

In [ ]:
if all_results:
    n_plots = min(3, len(all_results))
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    for idx, (species_name, results) in enumerate(list(all_results.items())[:3]):
        ax = axes[idx]

        sig = results[results['significant']]
        nonsig = results[~results['significant']]

        ax.scatter(nonsig['prevalence_diff'],
                   -np.log10(nonsig['q_value'].clip(lower=1e-50)),
                   c='gray', alpha=0.3, s=5, label=f'NS ({len(nonsig):,})')
        ax.scatter(sig['prevalence_diff'],
                   -np.log10(sig['q_value'].clip(lower=1e-50)),
                   c='red', alpha=0.6, s=10, label=f'Sig ({len(sig):,})')

        ax.axhline(-np.log10(0.05), color='red', linestyle='--', alpha=0.5)
        ax.set_xlabel('Max Prevalence Difference')
        ax.set_ylabel('-log10(q-value)')
        short_name = species_name.replace('s__', '').replace('_', ' ')
        ax.set_title(short_name, fontsize=9)
        ax.legend(fontsize=7)

    plt.tight_layout()
    plt.savefig('../figures/volcano_gene_env.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. COG enrichment visualization

In [ ]:
COG_NAMES = {
    'J': 'Translation', 'A': 'RNA processing', 'K': 'Transcription',
    'L': 'Replication/repair', 'B': 'Chromatin', 'D': 'Cell cycle',
    'Y': 'Nuclear', 'V': 'Defense', 'T': 'Signal transduction',
    'M': 'Cell wall', 'N': 'Cell motility', 'Z': 'Cytoskeleton',
    'W': 'Extracellular', 'U': 'Secretion', 'O': 'PTM/chaperones',
    'X': 'Mobilome', 'C': 'Energy', 'G': 'Carbohydrate',
    'E': 'Amino acid', 'F': 'Nucleotide', 'H': 'Coenzyme',
    'I': 'Lipid', 'P': 'Inorganic ion', 'Q': 'Secondary metabolites',
    'R': 'General prediction', 'S': 'Unknown function'
}

if enrichment_results:
    species_name = list(enrichment_results.keys())[0]
    data = enrichment_results[species_name]

    sig_frac = data['sig_cogs'] / data['sig_cogs'].sum()
    bg_frac = data['bg_cogs'] / data['bg_cogs'].sum()

    all_cogs = sorted(set(sig_frac.index) | set(bg_frac.index))
    cog_df = pd.DataFrame({
        'COG': all_cogs,
        'Env_associated': [sig_frac.get(c, 0) for c in all_cogs],
        'All_accessory': [bg_frac.get(c, 0) for c in all_cogs]
    })
    cog_df['enrichment'] = cog_df['Env_associated'] - cog_df['All_accessory']
    cog_df = cog_df.sort_values('enrichment', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    y_pos = range(len(cog_df))
    colors = ['#F44336' if e > 0 else '#2196F3' for e in cog_df['enrichment']]
    ax.barh(y_pos, cog_df['enrichment'], color=colors)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(
        [f"{row['COG']} - {COG_NAMES.get(row['COG'], '?')}" for _, row in cog_df.iterrows()],
        fontsize=8
    )
    ax.set_xlabel('Fraction Difference (env-associated - background)')
    short_name = species_name.replace('s__', '').replace('_', ' ')
    ax.set_title(f'COG Enrichment in Environment-Associated Genes\n{short_name}')
    ax.axvline(0, color='black', linewidth=0.5)

    plt.tight_layout()
    plt.savefig('../figures/cog_enrichment.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Cross-species summary

In [ ]:
summary_rows = []
for species_name, results in all_results.items():
    n_sig = results['significant'].sum()
    n_total = len(results)

    top_env = 'none'
    if n_sig > 0:
        top_env = results[results['significant']]['enriched_in'].value_counts().index[0]

    summary_rows.append({
        'species': species_name.replace('s__', ''),
        'clusters_tested': n_total,
        'significant_q05': n_sig,
        'pct_significant': round(100 * n_sig / n_total, 1) if n_total > 0 else 0,
        'top_enriched_env': top_env
    })

summary_df = pd.DataFrame(summary_rows).sort_values('significant_q05', ascending=False)
print("=== Cross-Species Summary ===")
print(summary_df.to_string(index=False))

# Save results
summary_df.to_csv('../data/gene_env_summary.csv', index=False)

for species_name, results in all_results.items():
    safe_name = species_name.replace('s__', '').replace(' ', '_')[:30]
    results.to_csv(f'../data/gene_env_{safe_name}.csv', index=False)

print(f"\nResults saved to data/ directory")

## 10. Methylobacterium descriptive analysis

*Methylobacterium* species don't meet the statistical testing criteria (>=10 genomes in >=2 environment categories), but a descriptive analysis of MDH type and B vitamin gene presence by environment is still informative. This connects the EC/KEGG-corrected MDH classification from notebook 01 to the environmental metadata from notebook 02 at the genome level.

In [ ]:
# --- Methylobacterium genomes with env metadata ---
methylo_genomes = genome_env[
    genome_env['gtdb_species_clade_id'].str.contains('Methylobacterium|Methylorubrum', na=False)
].copy()
print(f"Methylobacterium/Methylorubrum genomes with env metadata: {len(methylo_genomes)}")
print(f"\nEnvironment categories:")
print(methylo_genomes['env_category'].value_counts().to_string())

genome_ids = methylo_genomes['genome_id'].tolist()
genome_str = "','".join(genome_ids)

# --- MDH per genome (EC/KEGG-based, same as notebook 01) ---
refresh_spark()
print("\n--- Querying MDH annotations per genome ---")
mdh_per_genome = spark.sql(f"""
    SELECT DISTINCT
        g.genome_id,
        gc.gene_cluster_id,
        e.Preferred_name,
        e.EC,
        e.KEGG_ko
    FROM kbase_ke_pangenome.gene g
    JOIN kbase_ke_pangenome.gene_genecluster_junction ggj
        ON g.gene_id = ggj.gene_id
    JOIN kbase_ke_pangenome.gene_cluster gc
        ON ggj.gene_cluster_id = gc.gene_cluster_id
    JOIN kbase_ke_pangenome.eggnog_mapper_annotations e
        ON gc.gene_cluster_id = e.query_name
    WHERE gc.gtdb_species_clade_id IN (
        SELECT sc.gtdb_species_clade_id
        FROM kbase_ke_pangenome.gtdb_species_clade sc
        WHERE sc.GTDB_species LIKE '%Methylobacterium%'
           OR sc.GTDB_species LIKE '%Methylorubrum%'
    )
      AND g.genome_id IN ('{genome_str}')
      AND (e.Preferred_name IN ('xoxF', 'mxaF')
        OR e.KEGG_ko LIKE '%K14028%'
        OR e.KEGG_ko LIKE '%K00114%'
        OR e.EC LIKE '%1.1.2.7%'
        OR e.EC LIKE '%1.1.2.8%')
""").toPandas()
print(f"MDH gene-genome pairs: {len(mdh_per_genome)}")

def classify_mdh(row):
    """EC/KEGG classification (same as notebook 01)."""
    ec, kegg = str(row.get('EC', '')), str(row.get('KEGG_ko', ''))
    name = str(row.get('Preferred_name', ''))
    if '1.1.2.7' in ec or 'K14028' in kegg: return 'mxaF'
    if '1.1.2.8' in ec or 'K00114' in kegg: return 'xoxF'
    if name == 'xoxF': return 'xoxF'
    if name == 'mxaF': return 'mxaF'
    return 'ambiguous'

mdh_per_genome['mdh_type'] = mdh_per_genome.apply(classify_mdh, axis=1)
mdh_per_genome = mdh_per_genome[mdh_per_genome['mdh_type'] != 'ambiguous']

genome_mdh = mdh_per_genome.groupby('genome_id').agg(
    has_xoxF=('mdh_type', lambda x: (x == 'xoxF').any()),
    has_mxaF=('mdh_type', lambda x: (x == 'mxaF').any()),
    n_xoxF=('mdh_type', lambda x: (x == 'xoxF').sum()),
    n_mxaF=('mdh_type', lambda x: (x == 'mxaF').sum())
).reset_index()
genome_mdh['mdh_profile'] = genome_mdh.apply(
    lambda r: 'BOTH' if r['has_xoxF'] and r['has_mxaF']
    else ('xoxF_only' if r['has_xoxF'] else 'mxaF_only'), axis=1
)

methylo_env = methylo_genomes.merge(
    genome_mdh[['genome_id', 'mdh_profile', 'n_xoxF', 'n_mxaF']],
    on='genome_id', how='left'
)
methylo_env['mdh_profile'] = methylo_env['mdh_profile'].fillna('no_MDH')

print(f"\n=== MDH Profile x Environment Category ===")
ct = pd.crosstab(methylo_env['env_category'], methylo_env['mdh_profile'], margins=True)
print(ct.to_string())

# --- B vitamin genes per genome ---
refresh_spark()
print("\n--- Querying B vitamin genes per genome ---")
BVIT_PATHWAYS = {
    'B1': ['thiC', 'thiD', 'thiE', 'thiG', 'thiL', 'thiM', 'thiO', 'thiS'],
    'B2': ['ribA', 'ribB', 'ribC', 'ribD', 'ribE', 'ribH'],
    'B6': ['pdxA', 'pdxB', 'pdxJ', 'pdxH', 'pdxK'],
    'B7': ['bioA', 'bioB', 'bioC', 'bioD', 'bioF', 'bioH'],
    'B9': ['folA', 'folB', 'folC', 'folE', 'folK', 'folP', 'pabA', 'pabB'],
    'B12': ['cobA', 'cobB', 'cobC', 'cobD', 'cobE', 'cobG', 'cobH',
            'cobI', 'cobJ', 'cobK', 'cobL', 'cobM', 'cobN', 'cobO',
            'cobP', 'cobQ', 'cobR', 'cobS', 'cobT', 'cobU', 'cobW',
            'bluB', 'btuR']
}
all_bvit_genes = [g for genes in BVIT_PATHWAYS.values() for g in genes]
bvit_gene_str = "','".join(all_bvit_genes)

bvit_per_genome = spark.sql(f"""
    SELECT DISTINCT
        g.genome_id,
        e.Preferred_name
    FROM kbase_ke_pangenome.gene g
    JOIN kbase_ke_pangenome.gene_genecluster_junction ggj
        ON g.gene_id = ggj.gene_id
    JOIN kbase_ke_pangenome.gene_cluster gc
        ON ggj.gene_cluster_id = gc.gene_cluster_id
    JOIN kbase_ke_pangenome.eggnog_mapper_annotations e
        ON gc.gene_cluster_id = e.query_name
    WHERE gc.gtdb_species_clade_id IN (
        SELECT sc.gtdb_species_clade_id
        FROM kbase_ke_pangenome.gtdb_species_clade sc
        WHERE sc.GTDB_species LIKE '%Methylobacterium%'
           OR sc.GTDB_species LIKE '%Methylorubrum%'
    )
      AND g.genome_id IN ('{genome_str}')
      AND e.Preferred_name IN ('{bvit_gene_str}')
""").toPandas()
print(f"B vitamin gene-genome pairs: {len(bvit_per_genome)}")

# Map genes to pathways
gene_to_pathway = {}
for pathway, genes in BVIT_PATHWAYS.items():
    for g in genes:
        gene_to_pathway[g] = pathway

bvit_per_genome['pathway'] = bvit_per_genome['Preferred_name'].map(gene_to_pathway)

# Per-genome pathway completeness
genome_pathway_completeness = []
for genome_id in methylo_genomes['genome_id'].unique():
    genome_genes = set(bvit_per_genome[
        bvit_per_genome['genome_id'] == genome_id
    ]['Preferred_name'])
    row = {'genome_id': genome_id}
    for pathway, expected in BVIT_PATHWAYS.items():
        found = len(genome_genes & set(expected))
        row[f'{pathway}_found'] = found
        row[f'{pathway}_pct'] = round(100 * found / len(expected))
    row['total_bvit_genes'] = len(genome_genes & set(all_bvit_genes))
    genome_pathway_completeness.append(row)

completeness_df = pd.DataFrame(genome_pathway_completeness)
methylo_full = methylo_env.merge(completeness_df, on='genome_id', how='left')

print(f"\n=== B Vitamin Pathway Completeness by Environment ===")
pct_cols = [c for c in completeness_df.columns if c.endswith('_pct')]
for env in sorted(methylo_full['env_category'].unique()):
    subset = methylo_full[methylo_full['env_category'] == env]
    if len(subset) == 0:
        continue
    means = subset[pct_cols].mean()
    print(f"\n{env} (n={len(subset)}):")
    for col in pct_cols:
        pathway = col.replace('_pct', '')
        print(f"  {pathway}: {means[col]:.0f}% mean completeness")

print(f"\n=== B Vitamin Genes by MDH Profile ===")
for profile in ['BOTH', 'xoxF_only', 'mxaF_only', 'no_MDH']:
    subset = methylo_full[methylo_full['mdh_profile'] == profile]
    if len(subset) == 0:
        continue
    print(f"\n{profile} (n={len(subset)}):")
    print(f"  Mean total B vitamin genes: {subset['total_bvit_genes'].mean():.1f}")
    for col in pct_cols:
        pathway = col.replace('_pct', '')
        print(f"  {pathway}: {subset[col].mean():.0f}%")

## 11. Summary

### Gene-Environment Association Framework (S. aureus proof of concept)
- **S. aureus** (12,913 genomes across 5 environment categories): [TBD] significant gene-environment associations out of [TBD] accessory clusters tested
- The statistical framework (Fisher's exact / chi-squared, BH FDR correction) works and produces biologically interpretable results
- Full multi-species analysis deferred: queries joining `gene_genecluster_junction` for species with 10K+ genomes exceed the ~40-min Spark Connect auth timeout

### COG Enrichment
- [TBD after execution]

### Methylobacterium Descriptive Analysis
- MDH profile (EC/KEGG-based) x environment category: [TBD after execution]
- B vitamin pathway completeness by environment: [TBD after execution]
- B vitamin genes by MDH profile (genome-level): [TBD after execution]
- Sample sizes too small for formal statistical testing; patterns are descriptive only

### Limitations
- Environment categorization from free-text `isolation_source` is heuristic and lossy (e.g., "leaves" maps to "other" not "plant_associated")
- `ncbi_env` coverage is incomplete; many genomes lack environment metadata
- Spark Connect auth timeout (~40 min) limits single-query size; species with >10K genomes need query batching for the full multi-species analysis
- Methylobacterium analysis is descriptive due to small sample sizes per environment category

### Future Work
- Batch large-species queries (chunk genomes into groups of ~2000) to stay under auth timeout
- Extend to smaller species (Campylobacter, Enterococcus, Listeria — 1000-2000 genomes each) that can complete within timeout
- Improve environment categorization (map "leaves", "tree leaf surface" to plant_associated)